#### [Notebook 2 Quick Start](https://confluence.shopee.io/x/7O0lhw)
#### Common Use Cases and Solution Using Notebook 2
- [Multi Kernels in Notebook 2](https://confluence.shopee.io/x/4Od2hw)
- [Execute HDFS command](https://confluence.shopee.io/x/P5w8hw)
- [Browse, upload and download from Data Infra S3](https://confluence.shopee.io/x/xo9Ohw)
- [Setting up 3rd party packages for your Notebook 2](https://confluence.shopee.io/x/JZFOhw)
- [Develop and setup user defined function or library](https://confluence.shopee.io/x/hpjehg)
- [Execute Presto/Trino query directly in Python Kernel Notebook](https://confluence.shopee.io/x/WpMlhw)

> Note: "Python on K8S" Kernel cannot execute Spark functions. ***If you want to use Spark, change kernel to Spark***. [How to change Kernel](https://confluence.shopee.io/x/4Od2hw#MultiKernelsinNotebook2-SetKernelwhencreatingnewNotebook2file)

In [ ]:
import json
import sys
from datetime import datetime, date, timedelta
from typing import Dict, List, Any, Optional, Set

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, udf, lit, when, isnan, isnull, to_json, from_json,
    explode, collect_list, struct, array, size, coalesce,
    to_timestamp, datediff, current_date, regexp_extract, broadcast, get_json_object
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    FloatType, BooleanType, MapType, ArrayType, DoubleType
)
from pyspark.sql import functions as F
from itertools import product

import logging

# 配置日志
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def safe_to_float(x):
    try:
        if x is None:
            return None
        if isinstance(x, str):
            x = x.replace(',', '.')
        return float(x)
    except Exception:
        return None


def calculate_active_loan_features(active_loans: List[Dict], all_payment_dates: List, all_disburse_dates: List,
                                   current_date: datetime) -> Dict[str, Any]:
    """计算活跃贷款特征"""
    active_loan_count = len(active_loans)
    outstanding_amounts = [loan['outstandingBalance'] for loan in active_loans]

    total_outstanding = sum(outstanding_amounts) if outstanding_amounts else 0.0
    avg_outstanding = total_outstanding / active_loan_count if active_loan_count > 0 else 0.0
    max_outstanding = max(outstanding_amounts) if outstanding_amounts else 0.0
    min_outstanding = min(outstanding_amounts) if outstanding_amounts else 0.0

    # 计算未偿金额比率（相对于原始贷款金额）
    total_original_amount = sum(loan['contractAmount'] for loan in active_loans) if active_loans else 0.0
    outstanding_ratio = total_outstanding / total_original_amount if total_original_amount > 0 else 0.0

    # 日期计算 - 只计算活跃贷款的日期
    days_from_first_disburse = -9999
    days_from_last_disburse = -9999
    days_from_first_payment = -9999
    days_from_last_payment = -9999

    # 收集活跃贷款的放款日期
    active_disburse_dates = [loan['contractDate'] for loan in active_loans if loan['contractDate'] is not None]
    if active_disburse_dates:
        first_disburse = min(active_disburse_dates)
        last_disburse = max(active_disburse_dates)
        days_from_first_disburse = (current_date - first_disburse).days
        days_from_last_disburse = (current_date - last_disburse).days

    # 收集活跃贷款相关的还款日期（从all_payment_dates中筛选，只有活跃贷款有意义）
    if active_loan_count > 0 and all_payment_dates:
        first_payment = min(all_payment_dates)
        last_payment = max(all_payment_dates)
        days_from_first_payment = (current_date - first_payment).days
        days_from_last_payment = (current_date - last_payment).days

    # DPD计算（Days Past Due）
    current_max_dpd = max(loan['pastDueInstalments'] for loan in active_loans) if active_loans else 0
    historical_max_dpd = max(loan['pastDueInstalments'] for loan in active_loans) if active_loans else 0

    return {
        "inst_active_loan_cnt": active_loan_count,
        "inst_active_loan_os_amt_total": total_outstanding,
        "inst_active_loan_os_amt_avg": avg_outstanding,
        "inst_active_loan_os_amt_max": max_outstanding,
        "inst_active_loan_os_amt_min": min_outstanding,
        "inst_active_loan_os_amt_ratio": outstanding_ratio,
        "inst_active_loan_days_from_first_loan_disburse": round(days_from_first_disburse),
        "inst_active_loan_days_from_last_loan_disburse": round(days_from_last_disburse),
        "inst_active_loan_days_from_first_loan_repayment": round(days_from_first_payment),
        "inst_active_loan_days_from_last_loan_repayment": round(days_from_last_payment),
        "inst_active_loan_current_dpd_max": current_max_dpd,
        "inst_active_loan_historical_dpd_max": historical_max_dpd
    }


def calculate_closed_loan_features(closed_loans: List[Dict], all_payment_dates: List, all_disburse_dates: List,
                                   current_date: datetime, time_periods: Optional[List[str]] = None) -> Dict[str, Any]:
    """计算已关闭贷款特征（只统计带时间窗口的特征）"""
    features = {}

    # 必须指定时间周期
    if time_periods is None:
        return features

    # 有指定时间周期的情况
    periods = [int(p.replace('d', '')) for p in time_periods]
    period_strings = time_periods

    for i, days in enumerate(periods):
        period_str = period_strings[i]
        cutoff_date = datetime.fromtimestamp(current_date.timestamp() - days * 24 * 60 * 60)

        # 过滤在指定天数内关闭的贷款 - 使用lastPaymentDate
        recent_closed_loans = []
        for loan in closed_loans:
            # 优先使用lastPaymentDate来判断是否在时间窗口内
            if loan['lastPaymentDate'] and loan['lastPaymentDate'] > cutoff_date:
                recent_closed_loans.append(loan)

        # 计数特征
        features[f"inst_closed_loan_cnt_{period_str}"] = len(recent_closed_loans)

        # 金额特征
        amounts = [loan['contractAmount'] for loan in recent_closed_loans]
        features[f"inst_closed_loan_amt_total_{period_str}"] = sum(amounts) if amounts else 0.0
        features[f"inst_closed_loan_amt_avg_{period_str}"] = sum(amounts) / len(amounts) if amounts else 0.0
        features[f"inst_closed_loan_amt_max_{period_str}"] = max(amounts) if amounts else 0.0
        features[f"inst_closed_loan_amt_min_{period_str}"] = min(amounts) if amounts else 0.0

        # 历史最大逾期天数
        max_dpd = max(loan['pastDueInstalments'] for loan in recent_closed_loans) if recent_closed_loans else 0
        features[f"inst_closed_loan_historical_dpd_max_{period_str}"] = max_dpd

        # 日期特征 - 为每个时间周期都计算
        disburse_dates = [loan['contractDate'] for loan in recent_closed_loans if loan['contractDate'] is not None]
        # 使用每个贷款的最后还款日期
        last_payment_dates = [loan['lastPaymentDate'] for loan in recent_closed_loans if
                              loan['lastPaymentDate'] is not None]

        if disburse_dates:
            first_disburse = min(disburse_dates)
            last_disburse = max(disburse_dates)
            features[f"inst_closed_loan_days_from_first_loan_disburse_{period_str}"] = round(
                (current_date - first_disburse).days)
            features[f"inst_closed_loan_days_from_last_loan_disburse_{period_str}"] = round(
                (current_date - last_disburse).days)
        else:
            features[f"inst_closed_loan_days_from_first_loan_disburse_{period_str}"] = -9999
            features[f"inst_closed_loan_days_from_last_loan_disburse_{period_str}"] = -9999

        if last_payment_dates:
            first_payment = min(last_payment_dates)
            last_payment = max(last_payment_dates)
            features[f"inst_closed_loan_days_from_first_loan_repayment_{period_str}"] = round(
                (current_date - first_payment).days)
            features[f"inst_closed_loan_days_from_last_loan_repayment_{period_str}"] = round(
                (current_date - last_payment).days)
        else:
            features[f"inst_closed_loan_days_from_first_loan_repayment_{period_str}"] = -9999
            features[f"inst_closed_loan_days_from_last_loan_repayment_{period_str}"] = -9999

    return features


def calculate_active_loan_type_features(active_loans: List[Dict], current_date: datetime) -> Dict[str, Any]:
    """计算活跃贷款按类型分组特征"""
    features = {}

    # 按productType分组
    loans_by_type = {}
    for loan in active_loans:
        product_type = loan.get('productType')
        if product_type not in loans_by_type:
            loans_by_type[product_type] = []
        loans_by_type[product_type].append(loan)

    total_active_loans = len(active_loans)
    total_outstanding_amount = sum(loan['outstandingBalance'] for loan in active_loans) if active_loans else 0.0

    for product_type, loans in loans_by_type.items():
        loan_type = product_type.lower() if product_type else "unknown"

        # 数量特征
        type_count = len(loans)
        type_count_ratio = type_count / total_active_loans if total_active_loans > 0 else 0.0

        features[f"inst_active_loan_type_{loan_type}_cnt"] = type_count
        features[f"inst_active_loan_type_{loan_type}_cnt_ratio"] = type_count_ratio

        # 金额特征
        type_amounts = [loan['outstandingBalance'] for loan in loans]
        type_total_amount = sum(type_amounts) if type_amounts else 0.0
        type_avg_amount = type_total_amount / type_count if type_count > 0 else 0.0
        type_max_amount = max(type_amounts) if type_amounts else 0.0
        type_min_amount = min(type_amounts) if type_amounts else 0.0
        type_amount_ratio = type_total_amount / total_outstanding_amount if total_outstanding_amount > 0 else 0.0

        features[f"inst_active_loan_type_{loan_type}_os_amt_total"] = type_total_amount
        features[f"inst_active_loan_type_{loan_type}_os_amt_avg"] = type_avg_amount
        features[f"inst_active_loan_type_{loan_type}_os_amt_max"] = type_max_amount
        features[f"inst_active_loan_type_{loan_type}_os_amt_min"] = type_min_amount
        features[f"inst_active_loan_type_{loan_type}_os_amt_ratio"] = type_amount_ratio

    return features


def calculate_closed_loan_type_features(closed_loans: List[Dict], current_date: datetime,
                                        time_periods: Optional[List[str]] = None) -> Dict[str, Any]:
    """计算已关闭贷款按类型分组特征"""
    features = {}

    # 先收集所有存在的贷款类型
    all_loan_types = list(set(loan.get('productType', '').lower() for loan in closed_loans if loan.get('productType')))

    # 如果没有指定时间周期，统计全部已关闭贷款
    if time_periods is None:
        # 统计全部已关闭贷款，不限制时间
        recent_closed_loans = closed_loans

        # 按productType分组
        loans_by_type = {}
        for loan in recent_closed_loans:
            product_type = loan.get('productType')
            if product_type not in loans_by_type:
                loans_by_type[product_type] = []
            loans_by_type[product_type].append(loan)

        total_closed_loans = len(recent_closed_loans)
        total_closed_amount = sum(
            loan['contractAmount'] for loan in recent_closed_loans) if recent_closed_loans else 0.0

        # 为所有存在的贷款类型创建特征
        for loan_type in all_loan_types:
            loans = loans_by_type.get(loan_type.upper(), [])

            # 数量特征
            type_count = len(loans)
            type_count_ratio = type_count / total_closed_loans if total_closed_loans > 0 else 0.0

            features[f"inst_closed_loan_type_{loan_type}_cnt"] = type_count
            features[f"inst_closed_loan_type_{loan_type}_cnt_ratio"] = type_count_ratio

            # 金额特征
            type_amounts = [loan['contractAmount'] for loan in loans]
            type_total_amount = sum(type_amounts) if type_amounts else 0.0
            type_avg_amount = type_total_amount / type_count if type_count > 0 else 0.0
            type_max_amount = max(type_amounts) if type_amounts else 0.0
            type_min_amount = min(type_amounts) if type_amounts else 0.0
            type_amount_ratio = type_total_amount / total_closed_amount if total_closed_amount > 0 else 0.0

            features[f"inst_closed_loan_type_{loan_type}_amt_total"] = type_total_amount
            features[f"inst_closed_loan_type_{loan_type}_amt_avg"] = type_avg_amount
            features[f"inst_closed_loan_type_{loan_type}_amt_max"] = type_max_amount
            features[f"inst_closed_loan_type_{loan_type}_amt_min"] = type_min_amount
            features[f"inst_closed_loan_type_{loan_type}_amt_ratio"] = type_amount_ratio

        return features

    # 有指定时间周期的情况
    periods = [int(p.replace('d', '')) for p in time_periods]
    period_strings = time_periods

    for i, days in enumerate(periods):
        period_str = period_strings[i]
        cutoff_date = datetime.fromtimestamp(current_date.timestamp() - days * 24 * 60 * 60)

        # 过滤在指定天数内关闭的贷款 - 使用lastPaymentDate
        recent_closed_loans = []
        for loan in closed_loans:
            # 优先使用lastPaymentDate来判断是否在时间窗口内
            if loan['lastPaymentDate'] and loan['lastPaymentDate'] > cutoff_date:
                recent_closed_loans.append(loan)

        # 按productType分组
        loans_by_type = {}
        for loan in recent_closed_loans:
            product_type = loan.get('productType')
            if product_type not in loans_by_type:
                loans_by_type[product_type] = []
            loans_by_type[product_type].append(loan)

        total_closed_loans = len(recent_closed_loans)
        total_closed_amount = sum(
            loan['contractAmount'] for loan in recent_closed_loans) if recent_closed_loans else 0.0

        # 为所有存在的贷款类型创建特征，即使该时间窗口内没有该类型的贷款
        for loan_type in all_loan_types:
            loans = loans_by_type.get(loan_type.upper(), [])

            # 数量特征
            type_count = len(loans)
            type_count_ratio = type_count / total_closed_loans if total_closed_loans > 0 else 0.0

            features[f"inst_closed_loan_type_{loan_type}_cnt_{period_str}"] = type_count
            features[f"inst_closed_loan_type_{loan_type}_cnt_ratio_{period_str}"] = type_count_ratio

            # 金额特征
            type_amounts = [loan['contractAmount'] for loan in loans]
            type_total_amount = sum(type_amounts) if type_amounts else 0.0
            type_avg_amount = type_total_amount / type_count if type_count > 0 else 0.0
            type_max_amount = max(type_amounts) if type_amounts else 0.0
            type_min_amount = min(type_amounts) if type_amounts else 0.0
            type_amount_ratio = type_total_amount / total_closed_amount if total_closed_amount > 0 else 0.0

            features[f"inst_closed_loan_type_{loan_type}_amt_total_{period_str}"] = type_total_amount
            features[f"inst_closed_loan_type_{loan_type}_amt_avg_{period_str}"] = type_avg_amount
            features[f"inst_closed_loan_type_{loan_type}_amt_max_{period_str}"] = type_max_amount
            features[f"inst_closed_loan_type_{loan_type}_amt_min_{period_str}"] = type_min_amount
            features[f"inst_closed_loan_type_{loan_type}_amt_ratio_{period_str}"] = type_amount_ratio

    return features


def calculate_active_loan_frequency_features(active_loans: List[Dict], current_date: datetime) -> Dict[str, Any]:
    """计算活跃贷款按还款频率分组特征"""
    features = {}

    # 按instalmentPeriodicity分组
    loans_by_frequency = {}
    for loan in active_loans:
        pmt_freq_type = loan.get('instalmentPeriodicity')
        if pmt_freq_type not in loans_by_frequency:
            loans_by_frequency[pmt_freq_type] = []
        loans_by_frequency[pmt_freq_type].append(loan)

    total_active_loans = len(active_loans)
    total_outstanding_amount = sum(loan['outstandingBalance'] for loan in active_loans) if active_loans else 0.0

    for pmt_freq_type, loans in loans_by_frequency.items():
        freq_type = pmt_freq_type.lower() if pmt_freq_type else "unknown"

        # 数量特征
        freq_count = len(loans)
        freq_count_ratio = freq_count / total_active_loans if total_active_loans > 0 else 0.0

        features[f"inst_active_loan_pmt_freq_{freq_type}_cnt"] = freq_count
        features[f"inst_active_loan_pmt_freq_{freq_type}_cnt_ratio"] = freq_count_ratio

        # 金额特征
        freq_amounts = [loan['outstandingBalance'] for loan in loans]
        freq_total_amount = sum(freq_amounts) if freq_amounts else 0.0
        freq_avg_amount = freq_total_amount / freq_count if freq_count > 0 else 0.0
        freq_max_amount = max(freq_amounts) if freq_amounts else 0.0
        freq_min_amount = min(freq_amounts) if freq_amounts else 0.0
        freq_amount_ratio = freq_total_amount / total_outstanding_amount if total_outstanding_amount > 0 else 0.0

        features[f"inst_active_loan_pmt_freq_{freq_type}_os_amt_total"] = freq_total_amount
        features[f"inst_active_loan_pmt_freq_{freq_type}_os_amt_avg"] = freq_avg_amount
        features[f"inst_active_loan_pmt_freq_{freq_type}_os_amt_max"] = freq_max_amount
        features[f"inst_active_loan_pmt_freq_{freq_type}_os_amt_min"] = freq_min_amount
        features[f"inst_active_loan_pmt_freq_{freq_type}_os_amt_ratio"] = freq_amount_ratio

    return features


def calculate_closed_loan_frequency_features(closed_loans: List[Dict], current_date: datetime,
                                             time_periods: Optional[List[str]] = None) -> Dict[str, Any]:
    """计算已关闭贷款按还款频率分组特征"""
    features = {}

    # 先收集所有存在的还款频率类型
    all_freq_types = list(set(
        loan.get('instalmentPeriodicity', '').lower() for loan in closed_loans if loan.get('instalmentPeriodicity')))

    # 如果没有指定时间周期，统计全部已关闭贷款
    if time_periods is None:
        # 统计全部已关闭贷款，不限制时间
        recent_closed_loans = closed_loans

        # 按instalmentPeriodicity分组
        loans_by_frequency = {}
        for loan in recent_closed_loans:
            instalment_periodicity = loan.get('instalmentPeriodicity')
            if instalment_periodicity not in loans_by_frequency:
                loans_by_frequency[instalment_periodicity] = []
            loans_by_frequency[instalment_periodicity].append(loan)

        total_closed_loans = len(recent_closed_loans)
        total_closed_amount = sum(
            loan['contractAmount'] for loan in recent_closed_loans) if recent_closed_loans else 0.0

        # 为所有存在的频率类型创建特征
        for freq_type in all_freq_types:
            loans = loans_by_frequency.get(freq_type.upper(), [])

            # 数量特征
            freq_count = len(loans)
            freq_count_ratio = freq_count / total_closed_loans if total_closed_loans > 0 else 0.0

            features[f"inst_closed_loan_pmt_freq_{freq_type}_cnt"] = freq_count
            features[f"inst_closed_loan_pmt_freq_{freq_type}_cnt_ratio"] = freq_count_ratio

            # 金额特征
            freq_amounts = [loan['contractAmount'] for loan in loans]
            freq_total_amount = sum(freq_amounts) if freq_amounts else 0.0
            freq_avg_amount = freq_total_amount / freq_count if freq_count > 0 else 0.0
            freq_max_amount = max(freq_amounts) if freq_amounts else 0.0
            freq_min_amount = min(freq_amounts) if freq_amounts else 0.0
            freq_amount_ratio = freq_total_amount / total_closed_amount if total_closed_amount > 0 else 0.0

            features[f"inst_closed_loan_pmt_freq_{freq_type}_amt_total"] = freq_total_amount
            features[f"inst_closed_loan_pmt_freq_{freq_type}_amt_avg"] = freq_avg_amount
            features[f"inst_closed_loan_pmt_freq_{freq_type}_amt_max"] = freq_max_amount
            features[f"inst_closed_loan_pmt_freq_{freq_type}_amt_min"] = freq_min_amount
            features[f"inst_closed_loan_pmt_freq_{freq_type}_amt_ratio"] = freq_amount_ratio

        return features

    # 有指定时间周期的情况
    periods = [int(p.replace('d', '')) for p in time_periods]
    period_strings = time_periods

    for i, days in enumerate(periods):
        period_str = period_strings[i]
        cutoff_date = datetime.fromtimestamp(current_date.timestamp() - days * 24 * 60 * 60)

        # 过滤在指定天数内关闭的贷款 - 使用lastPaymentDate
        recent_closed_loans = []
        for loan in closed_loans:
            # 优先使用lastPaymentDate来判断是否在时间窗口内
            if loan['lastPaymentDate'] and loan['lastPaymentDate'] > cutoff_date:
                recent_closed_loans.append(loan)

        # 按instalmentPeriodicity分组
        loans_by_frequency = {}
        for loan in recent_closed_loans:
            instalment_periodicity = loan.get('instalmentPeriodicity')
            if instalment_periodicity not in loans_by_frequency:
                loans_by_frequency[instalment_periodicity] = []
            loans_by_frequency[instalment_periodicity].append(loan)

        total_closed_loans = len(recent_closed_loans)
        total_closed_amount = sum(
            loan['contractAmount'] for loan in recent_closed_loans) if recent_closed_loans else 0.0

        # 为所有存在的频率类型创建特征，即使该时间窗口内没有该频率的贷款
        for freq_type in all_freq_types:
            loans = loans_by_frequency.get(freq_type.upper(), [])

            # 数量特征
            freq_count = len(loans)
            freq_count_ratio = freq_count / total_closed_loans if total_closed_loans > 0 else 0.0

            features[f"inst_closed_loan_pmt_freq_{freq_type}_cnt_{period_str}"] = freq_count
            features[f"inst_closed_loan_pmt_freq_{freq_type}_cnt_ratio_{period_str}"] = freq_count_ratio

            # 金额特征
            freq_amounts = [loan['contractAmount'] for loan in loans]
            freq_total_amount = sum(freq_amounts) if freq_amounts else 0.0
            freq_avg_amount = freq_total_amount / freq_count if freq_count > 0 else 0.0
            freq_max_amount = max(freq_amounts) if freq_amounts else 0.0
            freq_min_amount = min(freq_amounts) if freq_amounts else 0.0
            freq_amount_ratio = freq_total_amount / total_closed_amount if total_closed_amount > 0 else 0.0

            features[f"inst_closed_loan_pmt_freq_{freq_type}_amt_total_{period_str}"] = freq_total_amount
            features[f"inst_closed_loan_pmt_freq_{freq_type}_amt_avg_{period_str}"] = freq_avg_amount
            features[f"inst_closed_loan_pmt_freq_{freq_type}_amt_max_{period_str}"] = freq_max_amount
            features[f"inst_closed_loan_pmt_freq_{freq_type}_amt_min_{period_str}"] = freq_min_amount
            features[f"inst_closed_loan_pmt_freq_{freq_type}_amt_ratio_{period_str}"] = freq_amount_ratio

    return features


# 常量定义
LOAN_TYPES = ['emprestimos', 'financiamentos']
PMT_FREQ_TYPES = [
    'sem_periodicidade_regular', 'semanal', 'quinzenal', 'mensal',
    'bimestral', 'trimestral', 'semestral', 'anual', 'outros'
]
AGGREGATION_TYPES = ['total', 'avg', 'max', 'min']
TIME_PERIODS = ['30d', '90d', '180d', '360d']


def generate_all_feature_names() -> Set[str]:
    """生成所有可能的特征名称"""
    features = set()

    # 活跃贷款基本特征
    features.add('inst_active_loan_cnt')
    # features.add('inst_active_loan_current_dpd_max')  # 注释：无法计算DPD
    # features.add('inst_active_loan_historical_dpd_max')  # 注释：无法计算DPD
    features.add('inst_active_loan_days_from_first_loan_disburse')
    features.add('inst_active_loan_days_from_last_loan_disburse')
    features.add('inst_active_loan_days_from_first_loan_repayment')
    features.add('inst_active_loan_days_from_last_loan_repayment')

    # 活跃贷款金额特征
    for agg_type in AGGREGATION_TYPES:
        features.add(f'inst_active_loan_os_amt_{agg_type}')
    features.add('inst_active_loan_os_amt_ratio')

    # 活跃贷款类型特征
    for loan_type in LOAN_TYPES:
        features.add(f'inst_active_loan_type_{loan_type}_cnt')
        features.add(f'inst_active_loan_type_{loan_type}_cnt_ratio')
        for agg_type in AGGREGATION_TYPES:
            features.add(f'inst_active_loan_type_{loan_type}_os_amt_{agg_type}')
        features.add(f'inst_active_loan_type_{loan_type}_os_amt_ratio')

    # 活跃贷款还款频率特征
    for freq_type in PMT_FREQ_TYPES:
        features.add(f'inst_active_loan_pmt_freq_{freq_type}_cnt')
        features.add(f'inst_active_loan_pmt_freq_{freq_type}_cnt_ratio')
        for agg_type in AGGREGATION_TYPES:
            features.add(f'inst_active_loan_pmt_freq_{freq_type}_os_amt_{agg_type}')
        features.add(f'inst_active_loan_pmt_freq_{freq_type}_os_amt_ratio')

    # 已关闭贷款基本特征（带时间窗口）
    for period in TIME_PERIODS:
        features.add(f'inst_closed_loan_cnt_{period}')
        # features.add(f'inst_closed_loan_historical_dpd_max_{period}')  # 注释：无法计算DPD
        features.add(f'inst_closed_loan_days_from_first_loan_disburse_{period}')
        features.add(f'inst_closed_loan_days_from_last_loan_disburse_{period}')
        features.add(f'inst_closed_loan_days_from_first_loan_repayment_{period}')
        features.add(f'inst_closed_loan_days_from_last_loan_repayment_{period}')

        # 已关闭贷款金额特征（带时间窗口）
        for agg_type in AGGREGATION_TYPES:
            features.add(f'inst_closed_loan_amt_{agg_type}_{period}')

    # 已关闭贷款类型特征（带时间窗口）
    for loan_type, period in product(LOAN_TYPES, TIME_PERIODS):
        features.add(f'inst_closed_loan_type_{loan_type}_cnt_{period}')
        features.add(f'inst_closed_loan_type_{loan_type}_cnt_ratio_{period}')
        for agg_type in AGGREGATION_TYPES:
            features.add(f'inst_closed_loan_type_{loan_type}_amt_{agg_type}_{period}')
        features.add(f'inst_closed_loan_type_{loan_type}_amt_ratio_{period}')

    # 已关闭贷款还款频率特征（带时间窗口）
    for freq_type, period in product(PMT_FREQ_TYPES, TIME_PERIODS):
        features.add(f'inst_closed_loan_pmt_freq_{freq_type}_cnt_{period}')
        features.add(f'inst_closed_loan_pmt_freq_{freq_type}_cnt_ratio_{period}')
        for agg_type in AGGREGATION_TYPES:
            features.add(f'inst_closed_loan_pmt_freq_{freq_type}_amt_{agg_type}_{period}')
        features.add(f'inst_closed_loan_pmt_freq_{freq_type}_amt_ratio_{period}')

    return features


def extract_contract_id_from_url(url: str) -> Optional[str]:
    """从URL中提取合同ID"""
    parts = url.split("/")

    # 首先尝试标准格式：查找"contracts"后的部分
    for i in range(len(parts)):
        if parts[i] == "contracts" and i + 1 < len(parts):
            return parts[i + 1]

    # 如果没找到"contracts"，尝试非标准格式
    # 查找以SPH、BCL等开头的契约ID（通常在v2后面）
    for i in range(len(parts)):
        if parts[i] == "v2" and i + 1 < len(parts):
            next_part = parts[i + 1]
            # 检查是否是契约ID格式（SPH或BCL开头）
            if next_part and (next_part.startswith("SPH") or next_part.startswith("BCL")):
                return next_part

    return None


def calculate_all_features(all_loans: List[Dict], all_payment_dates: List, all_disburse_dates: List,
                           current_date: datetime) -> Dict[str, Any]:
    """计算所有特征"""
    # 分离活跃贷款和已关闭贷款
    # 处理outstandingBalance可能为None的情况
    active_loans = [loan for loan in all_loans if
                    loan.get('outstandingBalance') is not None and loan['outstandingBalance'] > 0]
    closed_loans = [loan for loan in all_loans if
                    loan.get('outstandingBalance') is not None and loan['outstandingBalance'] == 0]

    features = {}

    # 计算所有特征
    features.update(calculate_active_loan_features(active_loans, all_payment_dates, all_disburse_dates, current_date))
    features.update(calculate_active_loan_type_features(active_loans, current_date))
    features.update(calculate_active_loan_frequency_features(active_loans, current_date))
    features.update(
        calculate_closed_loan_features(closed_loans, all_payment_dates, all_disburse_dates, current_date, TIME_PERIODS))
    features.update(calculate_closed_loan_type_features(closed_loans, current_date, TIME_PERIODS))
    features.update(calculate_closed_loan_frequency_features(closed_loans, current_date, TIME_PERIODS))

    return features


def calculate_active_loan_features(active_loans: List[Dict], all_payment_dates: List, all_disburse_dates: List,
                                   current_date: datetime) -> Dict[str, Any]:
    """计算活跃贷款特征"""
    active_loan_count = len(active_loans)
    outstanding_amounts = [loan['outstandingBalance'] for loan in active_loans]

    total_outstanding = sum(outstanding_amounts) if outstanding_amounts else 0.0
    avg_outstanding = total_outstanding / active_loan_count if active_loan_count > 0 else 0.0
    max_outstanding = max(outstanding_amounts) if outstanding_amounts else 0.0
    min_outstanding = min(outstanding_amounts) if outstanding_amounts else 0.0

    # 计算未偿金额比率（相对于原始贷款金额）
    total_original_amount = sum(loan['contractAmount'] for loan in active_loans) if active_loans else 0.0
    outstanding_ratio = total_outstanding / total_original_amount if total_original_amount > 0 else 0.0

    # 日期计算 - 只计算活跃贷款的日期
    days_from_first_disburse = -9999
    days_from_last_disburse = -9999
    days_from_first_payment = -9999
    days_from_last_payment = -9999

    # 收集活跃贷款的放款日期
    active_disburse_dates = [loan['contractDate'] for loan in active_loans if loan['contractDate'] is not None]
    if active_disburse_dates:
        first_disburse = min(active_disburse_dates)
        last_disburse = max(active_disburse_dates)
        days_from_first_disburse = (current_date - first_disburse).days
        days_from_last_disburse = (current_date - last_disburse).days

    # 收集活跃贷款相关的还款日期（从all_payment_dates中筛选，只有活跃贷款有意义）
    if active_loan_count > 0 and all_payment_dates:
        first_payment = min(all_payment_dates)
        last_payment = max(all_payment_dates)
        days_from_first_payment = (current_date - first_payment).days
        days_from_last_payment = (current_date - last_payment).days

    # DPD计算（Days Past Due）- 注释：无法计算DPD
    # current_max_dpd = max(loan['pastDueInstalments'] for loan in active_loans) if active_loans else 0
    # historical_max_dpd = max(loan['pastDueInstalments'] for loan in active_loans) if active_loans else 0

    return {
        "inst_active_loan_cnt": active_loan_count,
        "inst_active_loan_os_amt_total": total_outstanding,
        "inst_active_loan_os_amt_avg": avg_outstanding,
        "inst_active_loan_os_amt_max": max_outstanding,
        "inst_active_loan_os_amt_min": min_outstanding,
        "inst_active_loan_os_amt_ratio": outstanding_ratio,
        "inst_active_loan_days_from_first_loan_disburse": round(days_from_first_disburse),
        "inst_active_loan_days_from_last_loan_disburse": round(days_from_last_disburse),
        "inst_active_loan_days_from_first_loan_repayment": round(days_from_first_payment),
        "inst_active_loan_days_from_last_loan_repayment": round(days_from_last_payment),
        # "inst_active_loan_current_dpd_max": current_max_dpd,  # 注释：无法计算DPD
        # "inst_active_loan_historical_dpd_max": historical_max_dpd  # 注释：无法计算DPD
    }


def process_open_finance_data(open_finance_data: List[Dict], all_loans_map: Dict, all_payment_dates: List,
                              all_disburse_dates: List) -> None:
    """处理开放银行数据"""
    # 建立URL契约ID到合同编号的映射
    url_to_contract_mapping = {}

    for institution in open_finance_data:
        for api_call in institution['data']:
            url = api_call['url']
            rsp = api_call['rsp']

            # 如果rsp是字符串，先解析
            if isinstance(rsp, str):
                try:
                    # 处理转义字符问题
                    rsp_clean = rsp.encode().decode('unicode_escape')
                    rsp = json.loads(rsp_clean)
                    if isinstance(rsp, dict) and 'data' in rsp:
                        rsp = rsp['data']
                except (json.JSONDecodeError, TypeError, UnicodeDecodeError):
                    try:
                        # 如果上面失败，尝试直接解析
                        rsp = json.loads(rsp)
                        if isinstance(rsp, dict) and 'data' in rsp:
                            rsp = rsp['data']
                    except (json.JSONDecodeError, TypeError):
                        continue

            # 处理合同详情
            if ("/financings/" in url and "/contracts/" in url and
                    "/scheduled-instalments" not in url and "/payments" not in url):

                # 确保rsp是字典并包含必要字段
                if not isinstance(rsp, dict) or 'contractAmount' not in rsp:
                    continue

                contract_amount = float(rsp['contractAmount'])
                contract_date = datetime.strptime(rsp['contractDate'], '%Y-%m-%d')
                all_disburse_dates.append(contract_date)

                contract_id = rsp['contractNumber']
                url_contract_id = extract_contract_id_from_url(url)

                # 建立URL契约ID到合同编号的映射
                if url_contract_id:
                    url_to_contract_mapping[url_contract_id] = contract_id
                if contract_id not in all_loans_map:
                    loan_info = {
                        'contractId': contract_id,
                        'contractAmount': contract_amount,
                        'contractDate': contract_date,
                        'outstandingBalance': 0.0,
                        'pastDueInstalments': 0,
                        'productType': rsp.get('productType'),
                        'instalmentPeriodicity': rsp.get('instalmentPeriodicity'),
                        'lastPaymentDate': None,
                        'source': 'openFinance'
                    }
                    all_loans_map[contract_id] = loan_info

    # 第二遍处理：处理还款和分期信息，使用映射
    for institution in open_finance_data:
        for api_call in institution['data']:
            url = api_call['url']
            rsp = api_call['rsp']

            # 如果rsp是字符串，先解析
            if isinstance(rsp, str):
                try:
                    # 处理转义字符问题
                    rsp_clean = rsp.encode().decode('unicode_escape')
                    rsp = json.loads(rsp_clean)
                    if isinstance(rsp, dict) and 'data' in rsp:
                        rsp = rsp['data']
                except (json.JSONDecodeError, TypeError, UnicodeDecodeError):
                    try:
                        # 如果上面失败，尝试直接解析
                        rsp = json.loads(rsp)
                        if isinstance(rsp, dict) and 'data' in rsp:
                            rsp = rsp['data']
                    except (json.JSONDecodeError, TypeError):
                        continue

            # 处理还款信息 - 使用totalRemainingAmount
            if "/payments" in url:
                # 确保rsp是字典
                if not isinstance(rsp, dict):
                    continue

                url_contract_id = extract_contract_id_from_url(url)
                # 使用映射找到真实的合同编号
                contract_id = url_to_contract_mapping.get(url_contract_id, url_contract_id)

                # 优先使用totalRemainingAmount，如果没有则使用contractOutstandingBalance
                outstanding_balance = 0.0
                try:
                    if rsp.get('totalRemainingAmount') is not None:
                        result = safe_to_float(rsp['totalRemainingAmount'])
                        outstanding_balance = result if result is not None else 0.0
                    elif rsp.get('contractOutstandingBalance') is not None:
                        result = safe_to_float(rsp['contractOutstandingBalance'])
                        outstanding_balance = result if result is not None else 0.0
                except (ValueError, TypeError):
                    # 如果转换失败，保持默认值 0.0
                    outstanding_balance = 0.0

                # 更新未偿余额
                loan = all_loans_map.get(contract_id)
                if loan:
                    loan['outstandingBalance'] = outstanding_balance

                # 收集还款日期
                releases = rsp.get('releases', [])
                for release in releases:
                    if release.get('paidDate'):
                        paid_date = datetime.strptime(release['paidDate'], '%Y-%m-%d')
                        all_payment_dates.append(paid_date)
                        # 更新最后还款日期
                        if loan and (loan['lastPaymentDate'] is None or paid_date > loan['lastPaymentDate']):
                            loan['lastPaymentDate'] = paid_date

            # 处理分期信息（逾期情况）
            elif "/scheduled-instalments" in url:
                # 确保rsp是字典
                if not isinstance(rsp, dict):
                    continue

                url_contract_id = extract_contract_id_from_url(url)
                # 使用映射找到真实的合同编号
                contract_id = url_to_contract_mapping.get(url_contract_id, url_contract_id)
                past_due_instalments = rsp.get('pastDueInstalments', 0)

                loan = all_loans_map.get(contract_id)
                if loan:
                    loan['pastDueInstalments'] = max(loan['pastDueInstalments'], past_due_instalments)


def process_open_loan_data(open_loan_data: List[Dict], all_loans_map: Dict, all_payment_dates: List,
                           all_disburse_dates: List, debug: bool = False) -> None:
    """处理开放贷款数据"""
    # 建立URL契约ID到合同编号的映射
    url_to_contract_mapping = {}
    
    if debug:
        print("\n" + "="*80)
        print("📍 第一遍处理：建立合同映射（process_open_loan_data）")
        print("="*80)

    pass_count = 0
    for institution in open_loan_data:
        for api_call_idx, api_call in enumerate(institution['data'], 1):
            url = api_call['url']
            rsp = api_call['rsp']
            
            url_type = url.split('/')[-1] if '/' in url else url
            if debug:
                print(f"\n{api_call_idx}️⃣ {url_type}: {url}")

            # 如果rsp是字符串，先解析
            if isinstance(rsp, str):
                try:
                    # 处理转义字符问题
                    rsp_clean = rsp.encode().decode('unicode_escape')
                    rsp = json.loads(rsp_clean)
                    if isinstance(rsp, dict) and 'data' in rsp:
                        rsp = rsp['data']
                except (json.JSONDecodeError, TypeError, UnicodeDecodeError):
                    try:
                        # 如果上面失败，尝试直接解析
                        rsp = json.loads(rsp)
                        if isinstance(rsp, dict) and 'data' in rsp:
                            rsp = rsp['data']
                    except (json.JSONDecodeError, TypeError):
                        if debug:
                            print(f"   ⏭️  JSON 解析失败，跳过")
                        continue

            # 处理合同详情
            if ("/loans/" in url and "/contracts/" in url and
                    "/scheduled-instalments" not in url and "/payments" not in url and "/warranties" not in url):

                # 确保rsp是字典并包含必要字段
                if not isinstance(rsp, dict) or 'contractAmount' not in rsp:
                    if debug:
                        print(f"   ❌ 不是有效的合同详情（缺少 contractAmount）")
                    continue

                contract_amount = float(rsp['contractAmount'])
                contract_date = datetime.strptime(rsp['contractDate'], '%Y-%m-%d')
                all_disburse_dates.append(contract_date)

                contract_id = rsp['contractNumber']
                url_contract_id = extract_contract_id_from_url(url)

                if debug:
                    print(f"   ✅ 合同详情")
                    print(f"      - URL contractId: {url_contract_id}")
                    print(f"      - contractNumber: {contract_id}")
                    print(f"      - contractAmount: {contract_amount}")
                    print(f"      - contractDate: {contract_date.strftime('%Y-%m-%d')}")

                # 建立URL契约ID到合同编号的映射
                if url_contract_id:
                    url_to_contract_mapping[url_contract_id] = contract_id

                if contract_id not in all_loans_map:
                    loan_info = {
                        'contractId': contract_id,
                        'contractAmount': contract_amount,
                        'contractDate': contract_date,
                        'outstandingBalance': 0.0,
                        'pastDueInstalments': 0,
                        'productType': rsp.get('productType'),
                        'instalmentPeriodicity': rsp.get('instalmentPeriodicity'),
                        'lastPaymentDate': None,
                        'source': 'openLoan'
                    }
                    all_loans_map[contract_id] = loan_info
                    if debug:
                        print(f"      ✅ 创建贷款记录")
            else:
                if debug:
                    print(f"   ⏭️  不是 contracts detail，跳过")

    if debug:
        print(f"\n✅ 第一遍完成")
        print(f"   - URL 映射: {url_to_contract_mapping}")
        print(f"   - 贷款记录: {list(all_loans_map.keys())}")
        print("\n" + "="*80)
        print("📍 第二遍处理：处理还款和分期信息（process_open_loan_data）")
        print("="*80)

    # 第二遍：处理还款和分期信息
    pass_count = 0
    for institution in open_loan_data:
        for api_call_idx, api_call in enumerate(institution['data'], 1):
            url = api_call['url']
            rsp = api_call['rsp']
            
            url_type = url.split('/')[-1] if '/' in url else url

            # 如果rsp是字符串，先解析
            if isinstance(rsp, str):
                try:
                    # 处理转义字符问题
                    rsp_clean = rsp.encode().decode('unicode_escape')
                    rsp = json.loads(rsp_clean)
                    if isinstance(rsp, dict) and 'data' in rsp:
                        rsp = rsp['data']
                except (json.JSONDecodeError, TypeError, UnicodeDecodeError):
                    try:
                        # 如果上面失败，尝试直接解析
                        rsp = json.loads(rsp)
                        if isinstance(rsp, dict) and 'data' in rsp:
                            rsp = rsp['data']
                    except (json.JSONDecodeError, TypeError):
                        continue

            # 处理还款信息 - 也要检查开放贷款数据中的payments
            if "/payments" in url:
                if debug:
                    print(f"\n{api_call_idx}️⃣ {url_type}: {url}")
                    
                # 确保rsp是字典
                if not isinstance(rsp, dict):
                    if debug:
                        print(f"   ❌ rsp 不是字典")
                    continue

                url_contract_id = extract_contract_id_from_url(url)
                # 使用映射找到真实的合同编号
                contract_id = url_to_contract_mapping.get(url_contract_id, url_contract_id)
                
                if debug:
                    print(f"   💳 还款信息")
                    print(f"      - URL contractId: {url_contract_id}")
                    print(f"      - 映射后的 contractId: {contract_id}")

                # 优先使用totalRemainingAmount，如果没有则使用contractOutstandingBalance
                outstanding_balance = 0.0
                try:
                    if rsp.get('totalRemainingAmount') is not None:
                        result = safe_to_float(rsp['totalRemainingAmount'])
                        outstanding_balance = result if result is not None else 0.0
                    elif rsp.get('contractOutstandingBalance') is not None:
                        result = safe_to_float(rsp['contractOutstandingBalance'])
                        outstanding_balance = result if result is not None else 0.0
                except (ValueError, TypeError):
                    # 如果转换失败，保持默认值 0.0
                    outstanding_balance = 0.0

                # 更新未偿余额
                loan = all_loans_map.get(contract_id)
                if loan:
                    loan['outstandingBalance'] = outstanding_balance
                    if debug:
                        print(f"      ✅ 更新未偿余额: {outstanding_balance}")
                else:
                    if debug:
                        print(f"      ❌ 未找到贷款记录")

                # 收集还款日期
                releases = rsp.get('releases', [])
                if debug:
                    print(f"      📊 还款分期数: {len(releases)}")
                    
                for release in releases:
                    if release.get('paidDate'):
                        paid_date = datetime.strptime(release['paidDate'], '%Y-%m-%d')
                        all_payment_dates.append(paid_date)
                        # 更新最后还款日期
                        if loan and (loan['lastPaymentDate'] is None or paid_date > loan['lastPaymentDate']):
                            loan['lastPaymentDate'] = paid_date
                            if debug:
                                print(f"         → 更新 lastPaymentDate: {paid_date.strftime('%Y-%m-%d')}")

            # 处理分期信息（逾期情况）
            elif "/scheduled-instalments" in url:
                if debug:
                    print(f"\n{api_call_idx}️⃣ {url_type}: {url}")
                    
                # 确保rsp是字典
                if not isinstance(rsp, dict):
                    if debug:
                        print(f"   ❌ rsp 不是字典")
                    continue

                url_contract_id = extract_contract_id_from_url(url)
                # 使用映射找到真实的合同编号
                contract_id = url_to_contract_mapping.get(url_contract_id, url_contract_id)
                past_due_instalments = rsp.get('pastDueInstalments', 0)

                loan = all_loans_map.get(contract_id)
                if loan:
                    loan['pastDueInstalments'] = max(loan['pastDueInstalments'], past_due_instalments)
                    if debug:
                        print(f"   📋 分期信息")
                        print(f"      ✅ 更新 pastDueInstalments: {past_due_instalments}")
    
    if debug:
        print(f"\n✅ 第二遍完成")
        print(f"\n📋 最终贷款信息:")
        for cid, loan in all_loans_map.items():
            print(f"   {cid}:")
            print(f"      - 金额: {loan['contractAmount']}")
            print(f"      - 放款日期: {loan['contractDate'].strftime('%Y-%m-%d')}")
            print(f"      - 未偿余额: {loan['outstandingBalance']}")
            print(f"      - 最后还款日期: {loan['lastPaymentDate']}")


def get_cpf_loan_profile(cpf: str, get_open_finance_personal_financing_data: str,
                         get_open_finance_personal_loan_data: str, feature_list: Optional[List[str]] = None) -> Dict[
    str, Any]:
    """
    获取CPF贷款档案

    Args:
        cpf: CPF号码
        get_open_finance_personal_financing_data: 开放银行个人融资数据JSON字符串
        get_open_finance_personal_loan_data: 开放银行个人贷款数据JSON字符串
        feature_list: 可选的特征列表，如果提供则只返回指定的特征

    Returns:
        包含特征组的字典
    """
    # JSON解析
    loan_data = None
    financing_data = None
    try:
        if isinstance(get_open_finance_personal_financing_data, str):
            financing_data = json.loads(get_open_finance_personal_financing_data)
        else:
            financing_data = get_open_finance_personal_financing_data
        if isinstance(get_open_finance_personal_loan_data, str) and loan_data != '':
            loan_data = json.loads(get_open_finance_personal_loan_data)
        else:
            loan_data = get_open_finance_personal_loan_data
    except Exception as e:
        return {"cpf": cpf, "feature_group": {name: 0.0 for name in generate_all_feature_names()}}

    # 数据格式适配
    def extract_institution_data(data):
        if isinstance(data, dict):
            if 'get_open_finance_personal_financing_data' in data:
                return data['get_open_finance_personal_financing_data']
            elif 'get_open_finance_personal_loan_data' in data:
                return data['get_open_finance_personal_loan_data']
            elif 'data' in data:
                return data['data']
        elif isinstance(data, list):
            return data
        return []

    open_finance_data = extract_institution_data(financing_data)
    open_loan_data = extract_institution_data(loan_data)

    # 当前日期
    current_date = datetime.now()

    # 存储所有贷款信息 - 使用字典去重
    all_loans_map = {}
    all_payment_dates = []
    all_disburse_dates = []

    # 处理开放银行数据
    process_open_finance_data(open_finance_data, all_loans_map, all_payment_dates, all_disburse_dates)

    # 处理开放贷款数据（debug 模式）
    debug_mode = False  # 改为 True 可以启用调试信息
    process_open_loan_data(open_loan_data, all_loans_map, all_payment_dates, all_disburse_dates, debug=debug_mode)

    # 转换为列表
    all_loans = list(all_loans_map.values())

    # 计算特征指标
    features = calculate_all_features(all_loans, all_payment_dates, all_disburse_dates, current_date)

    # 获取所有可能的特征名称
    all_feature_names = generate_all_feature_names()

    # 如果没有提供feature_list，使用所有可能的特征
    if feature_list is None:
        feature_list = list(all_feature_names)

    # 构建最终的特征字典
    filtered_features = {}
    for feature_name in feature_list:
        if feature_name in features:
            filtered_features[feature_name] = features[feature_name]
        else:
            # 如果特征不存在，设置默认值
            filtered_features[feature_name] = 0.0  # 其他特征的默认值
    features = filtered_features

    return {
        "feature_group": features
    }


def read_source_data(ds, column1, column2, source_table="risk.dws_risk_br_openfinance_df"):
    """
    从源表读取数据
    Args:
        source_table: 源表名称
    Returns:
        DataFrame包含所需的列
    """

    df = spark.table(source_table)
    # 选择所需的列
    selected_df = df.select(
        col("shopee_user_id"),
        col("cpf"),
        col("request_date"),
        col("request_time"),
        col(f"{column1}"),
        col(f"{column2}")
    ).filter(
        (col("grass_date") == ds)
    )

    return selected_df


def write_to_hive(spark_df, category='', schema="dev_risk", table="ads_risk_br_openfinance_df", env="",
                  ds="2025-09-17"):
    """
    将 Spark DataFrame 写入 Hive 表 (支持分区覆盖)
    """

    root_path = f"/projects/risk/hdfs/{env}/risk_mart/ads/ads_risk_openfinance_df/grass_region=BR"
    # 添加分区字段 grass_date
    spark_df_with_partition = spark_df \
        .withColumn("grass_date", F.lit(ds)) \
        .withColumn("category", F.lit(category))

    # 覆盖写入 Hive (支持分区)
    (
        spark_df_with_partition
        .write.mode("overwrite")
        .format("parquet")
        .partitionBy("category", "grass_date")
        .option("partitionOverwriteMode", "dynamic")
        .option("parquet.compression", "snappy")  # 避免 zstd 兼容问题
        .option("spark.sql.parquet.writeLegacyFormat", "true")  # 避免 decimal/时间戳兼容
        .parquet(f"{root_path}")
    )

    print(f"✅ Data written to Hive table: {schema}.{table} partition grass_date={ds}")
    print(f"📊 Total records written: {spark_df_with_partition.count()}")

    # 如果 Hive Metastore 需要同步 HDFS 分区，可以在 Spark SQL 里调用 MSCK
    spark.sql(f"MSCK REPAIR TABLE {schema}.{table}")


def process_row(financing_data, loan_data, cpf):
    result = get_cpf_loan_profile(cpf, financing_data, loan_data)
    # 确保返回值格式正确
    feature_group = result.get("feature_group", {})
    # 将所有值转换为字符串以符合MapType定义
    feature_group_str = {k: f"{v}" if v is not None else None for k, v in feature_group.items()}

    feature_groups = {}
    feature_groups.update(feature_group_str)

    return {
        "feature_group": feature_groups
    }


def process_dataframe(input_df):
    """
    处理输入DataFrame

    Args:
        input_df: 输入DataFrame，包含shopee_user_id, cpf, get_open_finance_personal_account_data

    Returns:
        处理后的DataFrame，包含shopee_user_id, cpf, origin_data, feature_group
    """
    print("开始处理DataFrame...")

    # 检查输入数据
    input_count = input_df.count()
    print(f"输入数据记录数: {input_count}")
    # input_df.show(5, truncate=False)

    # 缓存输入DataFrame以提高性能
    input_df.cache()

    # 定义返回类型
    return_schema = StructType([
        StructField("feature_group", MapType(StringType(), StringType(), True), True)
    ])
    # 创建处理UDF
    process_udf = udf(process_row, return_schema)

    # 应用UDF处理数据，添加重分区以优化性能
    result_df = input_df.repartition(200).withColumn(
        "processed_result",
        process_udf(
            col("get_open_finance_personal_financing_data"),
            col("get_open_finance_personal_loan_data"),
            col("cpf")
        )
    )

    combine_df = result_df.withColumn("origin_data",
                                      to_json(
                                          struct(
                                              col("get_open_finance_personal_financing_data").alias("financing"),
                                              col("get_open_finance_personal_loan_data").alias("loan")
                                          )
                                      )
                                      )

    # 展开处理结果
    final_df = combine_df.select(
        col("shopee_user_id"),
        col("cpf"),
        col("request_date"),
        col("request_time"),
        to_json(col("processed_result.feature_group")).alias("feature_group"),
        col("origin_data").alias("origin_data"),

    )

    # 缓存结果DataFrame
    final_df.cache()

    # 检查输出数据质量
    output_count = final_df.count()
    print(f"输出数据记录数: {output_count}")

    if output_count != input_count:
        print(f"⚠️ 警告：输入输出记录数不匹配！输入: {input_count}, 输出: {output_count}")

    # 检查空值情况
    null_cpf_count = final_df.filter(col("cpf").isNull() | (col("cpf") == "")).count()
    null_feature_count = final_df.filter(col("feature_group").isNull()).count()

    print(f"📊 数据质量检查 - CPF空值: {null_cpf_count}, feature_group空值: {null_feature_count}")

    return final_df


def parse_arguments():
    """解析命令行参数"""
    import argparse

    yesterday = (date.today() - timedelta(days=1)).strftime("%Y-%m-%d")

    parser = argparse.ArgumentParser(description='CPF个人资料处理器')
    parser.add_argument('--source_table',
                        default='risk.dws_risk_br_openfinance_df',
                        help='源表名称 (默认: risk.dws_risk_br_openfinance_df)')

    parser.add_argument('--grass_date',
                        default=yesterday, )

    return parser.parse_args()

In [ ]:

args = parse_arguments()
spark = SparkSession.builder \
    .appName('app_name') \
    .master('yarn') \
    .getOrCreate()

df = read_source_data(ds='${ds}', column1="get_open_finance_personal_financing_data", column2="get_open_finance_personal_loan_data")
final_df = process_dataframe(df)
write_to_hive(final_df, category='loan_profile', schema='${SCHEMA}', table="ads_risk_br_openfinance_df",
              env='${ENV}', ds='${ds}')